# CV+ and CQR predictor preprocessing

CV+ and CQR predictions must use the preprocessing learned by the corresponding fold or quantile fit. The corrected implementation matches independent public-API references within 6.6 × 10⁻¹⁵ response units in all 72 deterministic cases. The preceding implementation differs in 39 cases, with a largest endpoint or point discrepancy of 0.126 response units.

The table reports every method, frontend, response family and stopping mode. These are shared-fixture numerical checks: 64 fixed-prefix cases and eight tuned cases. They do not estimate coverage or provide 72 independent sampling replicates. The responses are untransformed throughout this diagnostic. The [protocol](conformal_predictor_coding/PROTOCOL.md) was committed before implementation or measurement.

In [1]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
import mboostrs as mb
assert Path(mb.__file__).resolve().is_relative_to(root)
from benches.summarize_conformal_predictor_coding import load, validate
from benches.audit_conformal_predictor_coding import inputs

directory = root / "docs/book/benchmarks/conformal_predictor_coding"
rows = json.loads((directory / "summary.json").read_text())
facts = json.loads((directory / "validation.json").read_text())

In [2]:
comparison = pl.DataFrame([{
    "Response family": row["family"], "Frontend": row["frontend"].upper(),
    "Method": row["method"], "Stopping": row["budget"], "Cases": row["cases"],
    "Before": f'{row["baseline_max_error"]:.3e}',
    "Corrected": f'**{row["corrected_max_error"]:.3e}**',
    "Before mismatches": row["baseline_mismatches"],
    "Corrected mismatches": row["corrected_mismatches"]} for row in rows])
(GT(comparison, groupname_col="Response family")
 .tab_header(title="Predictor-coordinate accuracy against independent references",
             subtitle="Maximum absolute endpoint/point error in each family's response units; smaller is better")
 .tab_spanner(label="Numerical error", columns=["Before", "Corrected"])
 .tab_spanner(label="Cases exceeding 1e-10", columns=["Before mismatches", "Corrected mismatches"])
 .fmt_markdown(columns="Corrected")
 .tab_source_note(md("Bold errors satisfy the fixed 1e-10 tolerance. GLM/GAM denote glmboost/gamboost. "
    "Fixed stopping requests iteration 30 of a 60-step model. Tuned stopping uses five inner folds. "
    "Rows combine bare, scale, log and missing-predictor recipes and query/training-row targets; "
    "families are reported separately. These are deterministic identities, with no Monte Carlo SE.")))

GT(_tbl_data=shape: (12, 9)
┌────────────┬──────────┬────────┬──────────┬───┬───────────┬────────────┬────────────┬────────────┐
│ Response   ┆ Frontend ┆ Method ┆ Stopping ┆ … ┆ Before    ┆ Corrected  ┆ Before     ┆ Corrected  │
│ family     ┆ ---      ┆ ---    ┆ ---      ┆   ┆ ---       ┆ ---        ┆ mismatches ┆ mismatches │
│ ---        ┆ str      ┆ str    ┆ str      ┆   ┆ str       ┆ str        ┆ ---        ┆ ---        │
│ str        ┆          ┆        ┆          ┆   ┆           ┆            ┆ i64        ┆ i64        │
╞════════════╪══════════╪════════╪══════════╪═══╪═══════════╪════════════╪════════════╪════════════╡
│ gaussian   ┆ GLM      ┆ cv+    ┆ fixed    ┆ … ┆ 3.288e-02 ┆ **0.000e+0 ┆ 4          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 0**        ┆            ┆            │
│ gaussian   ┆ GLM      ┆ cv+    ┆ tuned    ┆ … ┆ 3.069e-02 ┆ **0.000e+0 ┆ 2          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 0**        ┆            ┆            │
│ gaussian   ┆ GLM      ┆ cqr    ┆ fixed    ┆ … ┆ 8.162e-02 ┆ **0.000e+0 ┆ 5          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 0**        ┆            ┆            │
│ gaussian   ┆ GLM      ┆ cqr    ┆ tuned    ┆ … ┆ 9.065e-02 ┆ **5.551e-1 ┆ 2          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 7**        ┆            ┆            │
│ gaussian   ┆ GAM      ┆ cv+    ┆ fixed    ┆ … ┆ 2.540e-02 ┆ **0.000e+0 ┆ 4          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 0**        ┆            ┆            │
│ …          ┆ …        ┆ …      ┆ …        ┆ … ┆ …         ┆ …          ┆ …          ┆ …          │
│ gaussian   ┆ GAM      ┆ cqr    ┆ tuned    ┆ … ┆ 9.043e-02 ┆ **6.509e-1 ┆ 2          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 5**        ┆            ┆            │
│ poisson    ┆ GLM      ┆ cv+    ┆ fixed    ┆ … ┆ 4.590e-02 ┆ **0.000e+0 ┆ 4          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 0**        ┆            ┆            │
│ poisson    ┆ GLM      ┆ cqr    ┆ fixed    ┆ … ┆ 0.000e+00 ┆ **0.000e+0 ┆ 0          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 0**        ┆            ┆            │
│ poisson    ┆ GAM      ┆ cv+    ┆ fixed    ┆ … ┆ 5.157e-02 ┆ **0.000e+0 ┆ 4          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 0**        ┆            ┆            │
│ poisson    ┆ GAM      ┆ cqr    ┆ fixed    ┆ … ┆ 1.260e-01 ┆ **0.000e+0 ┆ 5          ┆ 0          │
│            ┆          ┆        ┆          ┆   ┆           ┆ 0**        ┆            ┆            │
└────────────┴──────────┴────────┴──────────┴───┴───────────┴────────────┴────────────┴────────────┘, _body=<great_tables._gt_data.Body object at 0x10f48b1a0>, _boxhead=Boxhead([ColInfo(var='Response family', type=<ColInfoTypeEnum.row_group: 3>, column_label='Response family', column_align='left', column_width=None), ColInfo(var='Frontend', type=<ColInfoTypeEnum.default: 1>, column_label='Frontend', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Stopping', type=<ColInfoTypeEnum.default: 1>, column_label='Stopping', column_align='left', column_width=None), ColInfo(var='Cases', type=<ColInfoTypeEnum.default: 1>, column_label='Cases', column_align='right', column_width=None), ColInfo(var='Before', type=<ColInfoTypeEnum.default: 1>, column_label='Before', column_align='left', column_width=None), ColInfo(var='Corrected', type=<ColInfoTypeEnum.default: 1>, column_label='Corrected', column_align='left', column_width=None), ColInfo(var='Before mismatches', type=<ColInfoTypeEnum.default: 1>, column_label='Before mismatches', column_align='right', column_width=None), ColInfo(var='Corrected mismatches', type=<C

## Predictor recipes

A fitted predictor recipe stores values estimated from training rows, such as a mean and standard deviation for `scale(x)`, an imputation median, or a lower input boundary for `log(x)`. A refit on fewer rows estimates a new recipe. Its held-out predictions must replay that recipe on raw rows.

The diagnostic uses seed 636, 240 uniform[-1,1] latent predictors, and either Gaussian responses $0.3+0.6x+0.2Z$ or Poisson responses with mean $\exp(0.3+0.6x)$. For `log(x)`, the observed predictor is the exponential of the latent value. The missing case removes every thirteenth predictor and uses `na_action="impute"`. An unused row identifier traces the observations entering each fit. Query values correspond to latent predictors -0.6, 0 and 0.6; the missing case has its middle query predictor missing.

In [3]:
frame, query = inputs("gaussian", "scale")
model = mb.gamboost("y ~ bols(scale(x))", frame, control=mb.boost_control(60))
cv_plus = mb.conformal_interval(model, query, method="cv+", n_folds=4,
                               seed=4, mstop=30, level=.9)
cqr = mb.conformal_interval(model, query, method="cqr", seed=4, mstop=30, level=.9)

preview = pl.DataFrame({"x": query["x"], "CV+ lower": cv_plus.lower, "CV+ upper": cv_plus.upper,
                        "CQR lower": cqr.lower, "CQR upper": cqr.upper})
(GT(preview)
 .tab_header(title="Prediction bands with fold-specific predictor recipes",
             subtitle="Real endpoints in response units; nominal level 90%")
 .tab_spanner(label="CV+", columns=["CV+ lower", "CV+ upper"])
 .tab_spanner(label="CQR", columns=["CQR lower", "CQR upper"])
 .fmt_number(columns=["CV+ lower", "CV+ upper", "CQR lower", "CQR upper"], decimals=6)
 .tab_source_note("CV+ reports its applicable finite-sample lower bound separately; "
                  "the nominal level does not make that bound 90%. These three rows are an API example, "
                  "not a coverage experiment."))

GT(_tbl_data=shape: (3, 5)
┌──────┬───────────┬───────────┬───────────┬───────────┐
│ x    ┆ CV+ lower ┆ CV+ upper ┆ CQR lower ┆ CQR upper │
│ ---  ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│ f64  ┆ f64       ┆ f64       ┆ f64       ┆ f64       │
╞══════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ -0.6 ┆ -0.389784 ┆ 0.30257   ┆ -0.365815 ┆ 0.211698  │
│ 0.0  ┆ -0.04479  ┆ 0.647749  ┆ -0.044047 ┆ 0.559516  │
│ 0.6  ┆ 0.300204  ┆ 0.992743  ┆ 0.277721  ┆ 0.907334  │
└──────┴───────────┴───────────┴───────────┴───────────┘, _body=<great_tables._gt_data.Body object at 0x10ef87b00>, _boxhead=Boxhead([ColInfo(var='x', type=<ColInfoTypeEnum.default: 1>, column_label='x', column_align='right', column_width=None), ColInfo(var='CV+ lower', type=<ColInfoTypeEnum.default: 1>, column_label='CV+ lower', column_align='right', column_width=None), ColInfo(var='CV+ upper', type=<ColInfoTypeEnum.default: 1>, column_label='CV+ upper', column_align='right', column_width=None), ColInfo(var='CQR lower', type=<ColInfoTypeEnum.default: 1>, column_label='CQR lower', column_align='right', column_width=None), ColInfo(var='CQR upper', type=<ColInfoTypeEnum.default: 1>, column_label='CQR upper', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10fa14590>, _spanners=Spanners([SpannerInfo(spanner_id='CV+', spanner_level=0, spanner_label='CV+', spanner_units=None, spanner_pattern=None, vars=['CV+ lower', 'CV+ upper'], built=None), SpannerInfo(spanner_id='CQR', spanner_level=0, spanner_label='CQR', spanner_units=None, spanner_pattern=None, vars=['CQR lower', 'CQR upper'], built=None)]), _heading=Heading(title='Prediction bands with fold-specific predictor recipes', subtitle='Real endpoints in response units; nominal level 90%', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10fab7dd0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10fab7cb0>, _source_notes=['CV+ reports its applicable finite-sample lower bound separately; the nominal level does not make that bound 90%. These three rows are an API example, not a coverage experiment.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10fab44d0>, _formats=[<great_tables._gt_data.FormatInfo object at 0x10f4ec980>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, catego

## Reference fits and partitions

The CQR reference fits two quantile models directly on the raw proper-training half, predicts calibration and query rows through each model, and computes the corrected order statistic independently with NumPy. The CV+ reference directly fits each fold complement and combines the held residuals with that fold's query predictions. Reference fitting and prediction call the public model API, without calling private conformal prediction helpers.

All methods use split/fold seed 4. CQR reserves the first 120 permuted rows; CV+ has four held folds of 60 rows. Separate tests cover a 241-row input, its one set-aside row, one-row query chunks, a requested path beyond the original 60 steps, categorical encoding and below-training-range log inputs. Native conformal ranks, widening and CV+ guarantees are unchanged.

In [4]:
before, after = load("baseline"), load("corrected")
summary, verified = validate(before, after)
assert verified["cases"] == 72 and verified["corrected_mismatches"] == 0
assert verified["identical_references"] and verified["input_hashes_match"]
assert before["build_info"] == after["build_info"]

(GT(pl.DataFrame([{"Check": key.replace("_", " "), "Result": str(verified[key])}
    for key in ("cases", "baseline_mismatches", "corrected_mismatches", "maximum_corrected_error",
                "identical_references", "identical_native_binary", "input_hashes_match")]))
 .tab_header(title="Archive and reference validation", subtitle="Complete before/after records retained")
 .tab_source_note("Compressed and uncompressed SHA-256 digests are checked on load. "
                  "Reference arrays, partitions, selected stops and inner CV curves are identical across "
                  "the before/after diagnostic phases. Corrected prediction errors are recomputed from arrays."))

GT(_tbl_data=shape: (7, 2)
┌─────────────────────────┬──────────────────────┐
│ Check                   ┆ Result               │
│ ---                     ┆ ---                  │
│ str                     ┆ str                  │
╞═════════════════════════╪══════════════════════╡
│ cases                   ┆ 72                   │
│ baseline mismatches     ┆ 39                   │
│ corrected mismatches    ┆ 0                    │
│ maximum corrected error ┆ 6.50868248186498e-15 │
│ identical references    ┆ True                 │
│ identical native binary ┆ True                 │
│ input hashes match      ┆ True                 │
└─────────────────────────┴──────────────────────┘, _body=<great_tables._gt_data.Body object at 0x10f9cc080>, _boxhead=Boxhead([ColInfo(var='Check', type=<ColInfoTypeEnum.default: 1>, column_label='Check', column_align='left', column_width=None), ColInfo(var='Result', type=<ColInfoTypeEnum.default: 1>, column_label='Result', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10c2fbd40>, _spanners=Spanners([]), _heading=Heading(title='Archive and reference validation', subtitle='Complete before/after records retained', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x113cff6e0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x113cffe90>, _source_notes=['Compressed and uncompressed SHA-256 digests are checked on load. Reference arrays, partitions, selected stops and inner CV curves are identical across the before/after diagnostic phases. Corrected prediction errors are recomputed from arrays.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x10fbf42c0>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_bottom_style=OptionsInfo(scss=True, cat

## Inner stopping selection

With `mstop="tune"`, each proper-training fit is tuned using five inner folds. Preprocessing is re-estimated on each inner training fold. Calibration rows remain excluded from both the fitted quantile functions and the stopping decision.

The before/after evidence below records the actual CV mode, selected optima and grid lengths. The search retains the existing 1600-step starting floor, doubling rule, 10000-step cap and exact positive-grid argmin. The predictor repair does not introduce a new stopping criterion.

In [5]:
searches = []
for old, new in zip(before["results"], after["results"], strict=True):
    if new["budget"] != "tune":
        continue
    searches.append({"Frontend": new["frontend"].upper(), "Method": new["method"],
        "Predictor": new["recipe"], "Before mode": ", ".join(sorted({s["mode"] for s in old["inner_cv"]})),
        "Corrected mode": ", ".join(sorted({s["mode"] for s in new["inner_cv"]})),
        "Corrected stops": ", ".join(str(s["optimum"]) for s in new["inner_cv"]),
        "Reference stops": ", ".join(str(s["optimum"]) for s in new["reference"]["inner_cv"])})
(GT(pl.DataFrame(searches))
 .tab_header(title="Observed inner cross-validation", subtitle="Actual modes and selected iteration counts")
 .tab_source_note("Each row refers to the corresponding outer fit(s), in recorded order. "
                  "Both quantiles and every CV+ fold retain their own CV evidence. "
                  "One nearly flat curve has different exact argmins; its numerical effect is reported below."))

GT(_tbl_data=shape: (8, 7)
┌──────────┬────────┬───────────┬─────────────┬────────────────┬─────────────────┬─────────────────┐
│ Frontend ┆ Method ┆ Predictor ┆ Before mode ┆ Corrected mode ┆ Corrected stops ┆ Reference stops │
│ ---      ┆ ---    ┆ ---       ┆ ---         ┆ ---            ┆ ---             ┆ ---             │
│ str      ┆ str    ┆ str       ┆ str         ┆ str            ┆ str             ┆ str             │
╞══════════╪════════╪═══════════╪═════════════╪════════════════╪═════════════════╪═════════════════╡
│ GLM      ┆ cv+    ┆ scale     ┆ fixed       ┆ refit          ┆ 182, 66, 182,   ┆ 182, 66, 182,   │
│          ┆        ┆           ┆             ┆                ┆ 65              ┆ 65              │
│ GLM      ┆ cv+    ┆ missing   ┆ fixed       ┆ refit          ┆ 73, 55, 70, 38  ┆ 73, 55, 70, 38  │
│ GLM      ┆ cqr    ┆ scale     ┆ fixed       ┆ refit          ┆ 54, 58          ┆ 54, 58          │
│ GLM      ┆ cqr    ┆ missing   ┆ fixed       ┆ refit          ┆ 60, 53          ┆ 60, 53          │
│ GAM      ┆ cv+    ┆ scale     ┆ fixed       ┆ refit          ┆ 312, 66, 329,   ┆ 312, 66, 329,   │
│          ┆        ┆           ┆             ┆                ┆ 65              ┆ 65              │
│ GAM      ┆ cv+    ┆ missing   ┆ fixed       ┆ refit          ┆ 75, 56, 70, 39  ┆ 75, 56, 70, 39  │
│ GAM      ┆ cqr    ┆ scale     ┆ fixed       ┆ refit          ┆ 28, 30          ┆ 28, 30          │
│ GAM      ┆ cqr    ┆ missing   ┆ fixed       ┆ refit          ┆ 1556, 30        ┆ 76, 30          │
└──────────┴────────┴───────────┴─────────────┴────────────────┴─────────────────┴─────────────────┘, _body=<great_tables._gt_data.Body object at 0x113cff9e0>, _boxhead=Boxhead([ColInfo(var='Frontend', type=<ColInfoTypeEnum.default: 1>, column_label='Frontend', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Predictor', type=<ColInfoTypeEnum.default: 1>, column_label='Predictor', column_align='left', column_width=None), ColInfo(var='Before mode', type=<ColInfoTypeEnum.default: 1>, column_label='Before mode', column_align='left', column_width=None), ColInfo(var='Corrected mode', type=<ColInfoTypeEnum.default: 1>, column_label='Corrected mode', column_align='left', column_width=None), ColInfo(var='Corrected stops', type=<ColInfoTypeEnum.default: 1>, column_label='Corrected stops', column_align='left', column_width=None), ColInfo(var='Reference stops', type=<ColInfoTypeEnum.default: 1>, column_label='Reference stops', column_align='left', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10c31da00>, _spanners=Spanners([]), _heading=Heading(title='Observed inner cross-validation', subtitle='Actual modes and selected iteration counts', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x114069a90>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1140697c0>, _source_notes=['Each row refers to the corresponding outer fit(s), in recorded order. Both quantiles and every CV+ fold retain their own CV evidence. One nearly flat curve has different exact argmins; its numerical effect is reported below.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x114069760>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='v

## Numerical stopping sensitivity

One lower-quantile GAM fit with imputed predictors reports stop 1556, while the independent refit reports 76. Their inner risk curves differ by at most $5.55\times10^{-17}$, and the resulting interval discrepancy is $6.51\times10^{-15}$. Each stop is the exact positive-grid argmin of its own computed curve. This is a floating-point sensitivity on a nearly flat path, retained in [issue #653](https://github.com/fcela/mboostrs/issues/653). The diagnostic compares the curves within 1e-12 and the interval arrays within the prespecified 1e-10; it does not substitute a tolerance-based stopping rule after observing the difference.

In [6]:
sensitivity = verified["different_stop_labels_on_equivalent_curves"]
(GT(pl.DataFrame([{"Frontend": r["frontend"], "Method": r["method"], "Predictor": r["recipe"],
    "Actual stop": r["actual_stop"], "Reference stop": r["reference_stop"],
    "Risk difference": r["maximum_risk_difference"], "Interval error": r["interval_error"]}
    for r in sensitivity]))
 .tab_header(title="Retained nearly flat stopping curve", subtitle="Risk and response units; no sampling uncertainty")
 .fmt_scientific(columns=["Risk difference", "Interval error"], n_sigfig=4)
 .tab_source_note("The two selected stops obey the existing exact-minimum rule. "
                  "No candidate, grid, case or output was replaced to make the stop labels agree."))

GT(_tbl_data=shape: (1, 7)
┌──────────┬────────┬───────────┬─────────────┬────────────────┬─────────────────┬────────────────┐
│ Frontend ┆ Method ┆ Predictor ┆ Actual stop ┆ Reference stop ┆ Risk difference ┆ Interval error │
│ ---      ┆ ---    ┆ ---       ┆ ---         ┆ ---            ┆ ---             ┆ ---            │
│ str      ┆ str    ┆ str       ┆ i64         ┆ i64            ┆ f64             ┆ f64            │
╞══════════╪════════╪═══════════╪═════════════╪════════════════╪═════════════════╪════════════════╡
│ gam      ┆ cqr    ┆ missing   ┆ 1556        ┆ 76             ┆ 5.5511e-17      ┆ 6.5087e-15     │
└──────────┴────────┴───────────┴─────────────┴────────────────┴─────────────────┴────────────────┘, _body=<great_tables._gt_data.Body object at 0x10fab7ad0>, _boxhead=Boxhead([ColInfo(var='Frontend', type=<ColInfoTypeEnum.default: 1>, column_label='Frontend', column_align='left', column_width=None), ColInfo(var='Method', type=<ColInfoTypeEnum.default: 1>, column_label='Method', column_align='left', column_width=None), ColInfo(var='Predictor', type=<ColInfoTypeEnum.default: 1>, column_label='Predictor', column_align='left', column_width=None), ColInfo(var='Actual stop', type=<ColInfoTypeEnum.default: 1>, column_label='Actual stop', column_align='right', column_width=None), ColInfo(var='Reference stop', type=<ColInfoTypeEnum.default: 1>, column_label='Reference stop', column_align='right', column_width=None), ColInfo(var='Risk difference', type=<ColInfoTypeEnum.default: 1>, column_label='Risk difference', column_align='right', column_width=None), ColInfo(var='Interval error', type=<ColInfoTypeEnum.default: 1>, column_label='Interval error', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x114069460>, _spanners=Spanners([]), _heading=Heading(title='Retained nearly flat stopping curve', subtitle='Risk and response units; no sampling uncertainty', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x10fab4d70>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x10fbf4e30>, _source_notes=['The two selected stops obey the existing exact-minimum rule. No candidate, grid, case or output was replaced to make the stop labels agree.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x113cffb90>, _formats=[<great_tables._gt_data.FormatInfo object at 0x113cffc50>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=Options

## Exclusion checks and limits

Boundary tests inspect the actual row identifiers passed to outer and inner fits. They verify disjoint training/held-out observations, aligned case weights, and each model's recipe during prediction. Changing only CQR calibration responses and predictors leaves both quantile fits, their coefficients, predictions and selected stops identical; the final calibration correction changes. Existing numerical, CV+ remainder and guarantee tests remain in the validation suite.

The broader test run also exercised the stopping-cap warning behavior. When both quantiles reach the cap, one warning is reported per public call, including when the caller requests all warnings. Caller filters that convert the warning to an exception remain effective.

These checks establish predictor-coordinate consistency on the declared cases. They do not estimate coverage, conditional coverage, effect-inference validity or competitor performance. Response centering/scaling has a separate coordinate issue in standard models, tracked in [#652](https://github.com/fcela/mboostrs/issues/652); refitted inner CV explicitly refuses response transforms whose scales cannot be aggregated. This predictor-only repair does not change the units of response predictions. The independent Poisson benchmark records remain unchanged.

The [protocol](conformal_predictor_coding/PROTOCOL.md) and [validation report](conformal_predictor_coding/VALIDATION.md) document the scope, source/native identities and remaining limitations. [Issue #648](https://github.com/fcela/mboostrs/issues/648) tracks this correction. The archives retain complete outputs and inner CV evidence, including the differently selected stops.